
# 🌍 Portfolio Project — Real-Data Drought Risk Mapping for the MENA Region

**Author:** Ines Gasmi, PhD  
**Focus:** climate risk, geospatial analytics, machine learning, drought early warning  
**Core stack:** Google Earth Engine · geemap · pandas · matplotlib · scikit-learn · Folium  
**Data sources:** MODIS MOD13A3 NDVI · CHIRPS Daily rainfall · MODIS MOD11A2 land surface temperature

---

## Why this version matters

This notebook upgrades the prototype into a **real-data workflow**.  
Instead of simulated drought signals, it uses **satellite and climate observations** retrieved from Google Earth Engine and converts them into a clean machine-learning table.

### What this notebook demonstrates
- ingestion of real remote sensing data
- spatial aggregation over a MENA analysis grid
- time-series feature engineering
- explainable drought classification
- baseline machine-learning prediction
- interactive mapping for stakeholders

### Important note
This notebook is designed to be **portfolio-ready and methodologically honest**.  
It is a strong baseline for hiring because it shows an end-to-end workflow with real data before adding more advanced layers such as Sentinel-2, crop calendars, or XGBoost.



## Notebook Roadmap

1. Environment setup  
2. Define study area and spatial grid  
3. Load real MODIS, CHIRPS, and LST data from Earth Engine  
4. Build monthly grid-level dataset  
5. Engineer drought features  
6. Train a baseline Random Forest model  
7. Interpret results  
8. Create an interactive drought map  
9. Portfolio-ready next steps


---
## 1. Environment Setup

In [ ]:

# Run once if needed
# !pip install earthengine-api geemap folium pandas numpy matplotlib scikit-learn xgboost


In [ ]:

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

import folium

# Earth Engine / mapping
import ee
import geemap


In [ ]:

# Authenticate only the first time in a fresh environment.
# In Colab or Jupyter, uncomment the line below if needed:
# ee.Authenticate()

ee.Initialize()
print("Earth Engine initialized successfully.")



### Why these tools

- **Google Earth Engine** is the right tool for this stage because it handles large remote-sensing collections efficiently.
- **MODIS MOD13A3** gives monthly NDVI at a stable temporal frequency.
- **CHIRPS** adds rainfall, which is essential for drought interpretation.
- **MOD11A2** adds land surface temperature, which often strengthens drought-related signals.
- **Random Forest** is the right baseline because it is robust, explainable, and strong on tabular features.


---
## 2. Define Study Area and Spatial Grid

In [ ]:

# Analysis period
START_DATE = "2018-01-01"
END_DATE   = "2024-12-31"

# A broad MENA bounding box
# غرب/شرق/جنوب/شمال
mena_bbox = ee.Geometry.Rectangle([-17, 12, 65, 42])

# Grid size in degrees
GRID_DX = 1.0
GRID_DY = 1.0


In [ ]:

def build_fishnet(xmin=-17, ymin=12, xmax=65, ymax=42, dx=1.0, dy=1.0):
    """Create a simple rectangular grid as an Earth Engine FeatureCollection."""
    features = []
    cell_id = 0
    x = xmin
    while x < xmax:
        y = ymin
        while y < ymax:
            geom = ee.Geometry.Rectangle([x, y, min(x + dx, xmax), min(y + dy, ymax)])
            feat = ee.Feature(geom, {
                "cell_id": cell_id,
                "lon_min": x,
                "lat_min": y,
                "lon_max": min(x + dx, xmax),
                "lat_max": min(y + dy, ymax)
            })
            features.append(feat)
            cell_id += 1
            y += dy
        x += dx
    return ee.FeatureCollection(features)

grid_fc = build_fishnet(dx=GRID_DX, dy=GRID_DY)
print("Grid created.")


In [ ]:

# Quick interactive check
Map = geemap.Map(center=[28, 10], zoom=4)
Map.addLayer(mena_bbox, {"color": "black"}, "MENA bbox")
Map.addLayer(grid_fc.style(**{"color": "gray", "fillColor": "00000000", "width": 1}), {}, "Grid")
Map


---
## 3. Load Real Earth Observation Data


### Data choices

**NDVI:** `MODIS/061/MOD13A3`  
Monthly vegetation index, 1 km nominal resolution. NDVI is scaled by `0.0001`.

**Rainfall:** `UCSB-CHG/CHIRPS/DAILY`  
Daily precipitation, aggregated here to monthly totals.

**LST:** `MODIS/061/MOD11A2`  
8-day land surface temperature, aggregated to monthly mean. LST is scaled by `0.02` and converted from Kelvin to Celsius.


In [ ]:

# MODIS monthly NDVI
mod13a3 = (
    ee.ImageCollection("MODIS/061/MOD13A3")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(mena_bbox)
    .select("NDVI")
)

# CHIRPS daily rainfall
chirps = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(mena_bbox)
    .select("precipitation")
)

# MODIS LST (8-day)
mod11a2 = (
    ee.ImageCollection("MODIS/061/MOD11A2")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(mena_bbox)
    .select("LST_Day_1km")
)


In [ ]:

def month_starts(start, end):
    dates = pd.date_range(start=start, end=end, freq="MS")
    return [pd.Timestamp(d).strftime("%Y-%m-%d") for d in dates]

months = month_starts(START_DATE, END_DATE)
print(f"Months in analysis period: {len(months)}")
print(months[:3], "...", months[-3:])


In [ ]:

def build_monthly_image(month_start):
    """Create one multi-band monthly image: NDVI, rainfall, and LST."""
    start = ee.Date(month_start)
    end = start.advance(1, "month")

    ndvi_img = (
        mod13a3.filterDate(start, end).mean()
        .multiply(0.0001)
        .rename("ndvi")
    )

    rain_img = (
        chirps.filterDate(start, end).sum()
        .rename("rainfall")
    )

    lst_img = (
        mod11a2.filterDate(start, end).mean()
        .multiply(0.02)
        .subtract(273.15)
        .rename("lst_c")
    )

    month_num = ee.Number.parse(start.format("M"))
    year_num = ee.Number.parse(start.format("Y"))

    return (
        ee.Image.cat([ndvi_img, rain_img, lst_img])
        .clip(mena_bbox)
        .set({
            "system:time_start": start.millis(),
            "date": start.format("YYYY-MM-dd"),
            "year": year_num,
            "month": month_num
        })
    )

monthly_images = ee.ImageCollection.fromImages([build_monthly_image(m) for m in months])
print("Monthly image collection ready.")



## 4. Build a Monthly Grid-Level Dataset

Each monthly image is reduced over each grid cell to create one row per **cell-month**.

That gives a clean table like:

- `cell_id`
- `date`
- `ndvi`
- `rainfall`
- `lst_c`

This is the right bridge between remote sensing and machine learning.


In [ ]:

def reduce_month_to_grid(img, grid, scale=5000):
    """Reduce one monthly image over every grid cell."""
    reduced = img.reduceRegions(
        collection=grid,
        reducer=ee.Reducer.mean(),
        scale=scale
    )

    def add_meta(feat):
        return feat.set({
            "date": img.get("date"),
            "year": img.get("year"),
            "month": img.get("month")
        })

    return reduced.map(add_meta)

# Flatten all months into one FeatureCollection
monthly_fc = ee.FeatureCollection(
    monthly_images.map(lambda img: reduce_month_to_grid(ee.Image(img), grid_fc)).flatten()
)
print("Monthly grid-level feature collection created.")


In [ ]:

# Export to pandas DataFrame
records = monthly_fc.getInfo()["features"]

rows = []
for feat in records:
    props = feat["properties"]
    rows.append({
        "cell_id": props.get("cell_id"),
        "lon_min": props.get("lon_min"),
        "lat_min": props.get("lat_min"),
        "lon_max": props.get("lon_max"),
        "lat_max": props.get("lat_max"),
        "date": props.get("date"),
        "year": props.get("year"),
        "month": props.get("month"),
        "ndvi": props.get("ndvi"),
        "rainfall": props.get("rainfall"),
        "lst_c": props.get("lst_c"),
    })

df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["cell_id", "date"]).reset_index(drop=True)

print(df.shape)
df.head()



### Optional performance note

`getInfo()` is fine for a moderate portfolio demo, but it is **not** the best choice for very large production jobs.

For larger workflows, use:
- Earth Engine table export to Drive or Cloud Storage
- chunked regional exports
- direct ingestion into BigQuery or Parquet pipelines


In [ ]:

# Basic cleaning
df = df.dropna(subset=["ndvi", "rainfall", "lst_c"]).copy()

# Cell center coordinates for mapping
df["lon"] = (df["lon_min"] + df["lon_max"]) / 2
df["lat"] = (df["lat_min"] + df["lat_max"]) / 2

print("Rows after cleaning:", len(df))
df[["ndvi", "rainfall", "lst_c"]].describe().round(3)


---
## 5. Feature Engineering and Drought Labels

In [ ]:

# Monthly climatology by cell
df["month"] = df["date"].dt.month
df["year"]  = df["date"].dt.year

cell_month_clim = (
    df.groupby(["cell_id", "month"])["ndvi"]
    .mean()
    .rename("ndvi_climatology")
    .reset_index()
)

df = df.merge(cell_month_clim, on=["cell_id", "month"], how="left")
df["ndvi_anomaly"] = df["ndvi"] - df["ndvi_climatology"]

# Rolling features
df = df.sort_values(["cell_id", "date"]).copy()

for col, win, new_col in [
    ("ndvi_anomaly", 3, "ndvi_anom_3m"),
    ("ndvi_anomaly", 6, "ndvi_anom_6m"),
    ("rainfall", 3, "rainfall_3m"),
    ("lst_c", 3, "lst_3m"),
]:
    df[new_col] = (
        df.groupby("cell_id")[col]
        .transform(lambda s: s.rolling(win, min_periods=1).mean())
    )

# Rainfall anomaly
rain_clim = (
    df.groupby(["cell_id", "month"])["rainfall"]
    .mean()
    .rename("rainfall_climatology")
    .reset_index()
)
df = df.merge(rain_clim, on=["cell_id", "month"], how="left")
df["rainfall_anomaly"] = df["rainfall"] - df["rainfall_climatology"]

df.head()


In [ ]:

def classify_drought_from_ndvi(anom):
    """Simple explainable drought classes based on NDVI anomaly."""
    if pd.isna(anom):
        return np.nan
    if anom <= -0.14:
        return 3  # Severe
    elif anom <= -0.08:
        return 2  # Moderate
    elif anom <= -0.04:
        return 1  # Mild
    else:
        return 0  # Normal / Wet

df["drought_class"] = df["ndvi_anomaly"].apply(classify_drought_from_ndvi)

class_names = {
    0: "Normal / Wet",
    1: "Mild Drought",
    2: "Moderate Drought",
    3: "Severe Drought",
}

df["drought_label"] = df["drought_class"].map(class_names)
df["drought_class"].value_counts(dropna=False).sort_index()



### Why this label choice is acceptable

For a first real-data version, using an **anomaly-based rule** is acceptable because it is:
- transparent
- easy to explain
- aligned with vegetation stress logic

Later, labels can be strengthened with:
- SPI / SPEI
- crop-specific stress thresholds
- external drought event inventories


---
## 6. Baseline Machine Learning Model

In [ ]:

model_df = df.dropna(subset=[
    "ndvi", "ndvi_anomaly", "ndvi_anom_3m", "ndvi_anom_6m",
    "rainfall", "rainfall_3m", "rainfall_anomaly",
    "lst_c", "lst_3m", "drought_class"
]).copy()

FEATURES = [
    "ndvi",
    "ndvi_anomaly",
    "ndvi_anom_3m",
    "ndvi_anom_6m",
    "rainfall",
    "rainfall_3m",
    "rainfall_anomaly",
    "lst_c",
    "lst_3m",
    "month",
    "lat",
    "lon"
]
TARGET = "drought_class"

X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


In [ ]:

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=[class_names[i] for i in sorted(class_names)]))


In [ ]:

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=[class_names[i] for i in sorted(class_names)],
    cmap="Blues",
    ax=ax,
    xticks_rotation=30
)
ax.set_title("Random Forest — Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:

feat_imp = (
    pd.Series(rf.feature_importances_, index=FEATURES)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 5))
feat_imp.plot(kind="bar", ax=ax)
ax.set_title("Feature Importance — Random Forest")
ax.set_ylabel("Importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

feat_imp.round(4)



## 7. Results and Interpretation

At this stage, the project already demonstrates the most important hiring signal:

> the ability to turn raw earth observation data into an interpretable machine-learning workflow.

### What to look for in the results
- whether `ndvi_anomaly` and `ndvi_anom_3m` dominate the model
- whether rainfall features add predictive value
- whether temperature strengthens drought severity separation
- whether class confusion happens mainly between mild and moderate drought

That is a normal and believable pattern in real drought classification.


In [ ]:

# Annual drought shares
annual = (
    model_df.groupby(["year", "drought_label"])
    .size()
    .unstack(fill_value=0)
)

annual_pct = annual.div(annual.sum(axis=1), axis=0) * 100
annual_pct = annual_pct[[c for c in class_names.values() if c in annual_pct.columns]]

fig, ax = plt.subplots(figsize=(11, 5))
annual_pct.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Annual Drought Severity Share Across the MENA Grid")
ax.set_ylabel("Percent of grid-cell months")
ax.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:

# Relationship between rainfall anomaly and NDVI anomaly
sample_df = model_df.sample(min(5000, len(model_df)), random_state=42)

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    sample_df["rainfall_anomaly"],
    sample_df["ndvi_anomaly"],
    c=sample_df["drought_class"],
    alpha=0.4
)
ax.set_title("NDVI Anomaly vs Rainfall Anomaly")
ax.set_xlabel("Rainfall anomaly")
ax.set_ylabel("NDVI anomaly")
plt.tight_layout()
plt.show()


---
## 8. Interactive Drought Map

In [ ]:

# Select latest month for mapping
latest_date = model_df["date"].max()
latest_df = model_df[model_df["date"] == latest_date].copy()

# Predict model class on latest month
latest_df["predicted_class"] = rf.predict(latest_df[FEATURES])
latest_df["predicted_label"] = latest_df["predicted_class"].map(class_names)

color_map = {
    0: "#2E86AB",  # Normal / Wet
    1: "#F18F01",  # Mild
    2: "#E76F51",  # Moderate
    3: "#C73E1D",  # Severe
}

m = folium.Map(location=[28, 10], zoom_start=4, tiles="CartoDB positron")

for _, row in latest_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        color=color_map[row["predicted_class"]],
        fill=True,
        fill_opacity=0.8,
        popup=(
            f"Cell {int(row['cell_id'])}<br>"
            f"Date: {row['date'].date()}<br>"
            f"NDVI: {row['ndvi']:.3f}<br>"
            f"Rainfall: {row['rainfall']:.1f}<br>"
            f"LST °C: {row['lst_c']:.1f}<br>"
            f"Class: {row['predicted_label']}"
        )
    ).add_to(m)

m



This map is intentionally simple and recruiter-friendly.  
It shows that the project can move from:
- satellite ingestion
- to tabular ML
- to stakeholder-facing spatial communication

That is a much stronger portfolio signal than adding deep learning too early.


---
## 9. Portfolio-Ready Next Steps


### Strong next upgrades in the right order

1. **Benchmark Random Forest against XGBoost**  
   Improve the tabular baseline before exploring complex sequence models.

2. **Replace the broad bounding box with real administrative or agro-ecological boundaries**  
   This will make the output more policy-relevant.

3. **Integrate crop calendar data**  
   Weight drought severity by the growing season instead of treating all months equally.

4. **Add Sentinel-2 in selected agricultural hotspots**  
   Use it where higher spatial resolution creates real value.

5. **Export maps and tables as reusable portfolio assets**  
   Save figures, HTML maps, and clean CSV outputs for GitHub and presentations.

6. **Only then consider LSTM or temporal deep learning**  
   Deep learning should be an extension, not the first proof of competence.



## Final Portfolio Positioning

A strong way to describe this project is:

> **Built an end-to-end drought risk mapping workflow for the MENA region using real MODIS, CHIRPS, and LST data from Google Earth Engine, engineered monthly geospatial features, trained a Random Forest classifier, and produced stakeholder-friendly interactive maps.**

That statement is clear, credible, and valuable for hiring.
